In [0]:
# ============================================================
# NOTEBOOK 02b — EDA WITH SparkSQL
# Vermont School - Early Warning System V2
#
# INPUT:  bronze/prepared/24_25/ and bronze/prepared/25_26/
# OUTPUT: silver/eda_* (Parquet)
# Cataloging: Spark temporary views (Databricks Unity Catalog)
# ============================================================

BRONZE = "/Volumes/workspace/vermont/bronze"
SILVER = "/Volumes/workspace/vermont/silver"

PREP_24 = f"{BRONZE}/prepared/24_25"
PREP_25 = f"{BRONZE}/prepared/25_26"

# Load and register as catalog tables
df_24 = spark.read.parquet(PREP_24)
df_25 = spark.read.parquet(PREP_25)
df_all = df_24.union(df_25)

df_24.createOrReplaceTempView("students_24_25")
df_25.createOrReplaceTempView("students_25_26")
df_all.createOrReplaceTempView("students_all")

print("✓ Tables registered in Spark catalog:")
print("  - students_24_25")
print("  - students_25_26")
print("  - students_all")
print(f"\n  24-25: {df_24.count()} students")
print(f"  25-26: {df_25.count()} students")
print(f"  Combined: {df_all.count()} records")

# Verify catalog
spark.sql("SHOW VIEWS").show()

In [0]:
# CELDA 2 — EDA: Risk distribution analysis

print("=" * 60)
print("EDA WITH SparkSQL — Vermont School")
print("=" * 60)

# Query 1: Risk distribution by grade and year
q1 = spark.sql("""
    SELECT 
        year,
        grade,
        risk_level,
        COUNT(*) as n_students,
        ROUND(COUNT(*) * 100.0 / 
              SUM(COUNT(*)) OVER (PARTITION BY year, grade), 1) as pct_in_grade
    FROM students_all
    GROUP BY year, grade, risk_level
    ORDER BY year, grade, risk_level
""")
print("\n── Q1: Risk distribution by grade and year ──")
q1.show(30, truncate=False)
q1.write.mode("overwrite").parquet(f"{SILVER}/eda_risk_by_grade_year")

# Query 2: Academic performance by grade
q2 = spark.sql("""
    SELECT
        year,
        grade,
        COUNT(*) as n_students,
        ROUND(AVG(avg_cumulative), 2)      as avg_grade,
        ROUND(MIN(min_cumulative), 2)      as min_grade,
        ROUND(AVG(n_subjects_below_4), 2)  as avg_subjects_below_4,
        ROUND(AVG(total_absences), 1)      as avg_absences,
        ROUND(AVG(n_f1), 2)                as avg_f1,
        MAX(n_f1)                          as max_f1
    FROM students_all
    GROUP BY year, grade
    ORDER BY year, grade
""")
print("\n── Q2: Academic performance by grade and year ──")
q2.show(truncate=False)
q2.write.mode("overwrite").parquet(f"{SILVER}/eda_performance_by_grade")

# Query 3: Subject performance — which subjects fail most
q3 = spark.sql("""
    SELECT
        year,
        ROUND(AVG(Science_cum), 2)              as Science,
        ROUND(AVG(Mathematics_cum), 2)           as Mathematics,
        ROUND(AVG(English_cum), 2)               as English,
        ROUND(AVG(Lengua_Castellana_cum), 2)     as Lengua_Castellana,
        ROUND(AVG(Mandarin_cum), 2)              as Mandarin,
        ROUND(AVG(Financial_Maths_cum), 2)       as Financial_Maths,
        ROUND(AVG(ICT_STEM_cum), 2)              as ICT_STEM,
        ROUND(AVG(Physical_Education_cum), 2)    as Physical_Education,
        ROUND(AVG(I_and_S_cum), 2)               as I_and_S,
        ROUND(AVG(Research_Methodology_cum), 2)  as Research_Methodology
    FROM students_all
    GROUP BY year
    ORDER BY year
""")
print("\n── Q3: Average grade by subject and year ──")
q3.show(truncate=False)
q3.write.mode("overwrite").parquet(f"{SILVER}/eda_subject_performance")

print("✓ All EDA results saved to Silver")

In [0]:
# CELDA 3 — EDA: Correlation and absences analysis

# Query 4: Absences vs risk level
q4 = spark.sql("""
    SELECT
        year,
        risk_level,
        COUNT(*) as n_students,
        ROUND(AVG(total_absences), 1)   as avg_absences,
        ROUND(AVG(absence_class), 1)    as avg_absence_class,
        ROUND(AVG(late), 1)             as avg_late,
        ROUND(AVG(n_f1), 2)             as avg_f1,
        ROUND(AVG(n_f2), 2)             as avg_f2,
        ROUND(AVG(avg_cumulative), 2)   as avg_grade
    FROM students_all
    GROUP BY year, risk_level
    ORDER BY year,
        CASE risk_level
            WHEN 'critical'  THEN 1
            WHEN 'recovery'  THEN 2
            WHEN 'no_risk'   THEN 3
        END
""")
print("── Q4: Absences and disciplinary by risk level ──")
q4.show(truncate=False)
q4.write.mode("overwrite").parquet(f"{SILVER}/eda_absences_vs_risk")

# Query 5: Year over year comparison
q5 = spark.sql("""
    SELECT
        year,
        COUNT(*) as total_students,
        SUM(CASE WHEN risk_level='critical'  THEN 1 ELSE 0 END) as critical,
        SUM(CASE WHEN risk_level='recovery'  THEN 1 ELSE 0 END) as recovery,
        SUM(CASE WHEN risk_level='no_risk'   THEN 1 ELSE 0 END) as no_risk,
        ROUND(AVG(avg_cumulative), 2)    as avg_grade,
        ROUND(AVG(total_absences), 1)    as avg_absences,
        ROUND(AVG(n_f1), 2)              as avg_f1
    FROM students_all
    GROUP BY year
    ORDER BY year
""")
print("\n── Q5: Year over year comparison ──")
q5.show(truncate=False)
q5.write.mode("overwrite").parquet(f"{SILVER}/eda_year_comparison")

# Visualizations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

q4_pd = q4.toPandas()
q5_pd = q5.toPandas()
q1_pd = q1.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Vermont School — EDA with SparkSQL\n24-25 vs 25-26',
             fontweight='bold', fontsize=13)

colors_risk = {'critical': '#e74c3c', 'recovery': '#f39c12', 'no_risk': '#2ecc71'}

# Plot 1: Risk distribution by grade — both years
for idx, year in enumerate(['24_25', '25_26']):
    ax = axes[0, idx]
    df_y = q1_pd[q1_pd['year'] == year]
    grades = sorted(df_y['grade'].unique())
    x = range(len(grades))
    width = 0.25
    for i, risk in enumerate(['critical', 'recovery', 'no_risk']):
        vals = [df_y[(df_y['grade']==g) & 
                     (df_y['risk_level']==risk)]['n_students'].sum() 
                for g in grades]
        ax.bar([xi + i*width for xi in x], vals,
               width=width, color=colors_risk[risk],
               label=risk.replace('_',' ').title())
    ax.set_xticks([xi + width for xi in x])
    ax.set_xticklabels([f'Grade {g}°' for g in grades])
    ax.set_title(f'Risk by Grade — {year.replace("_","-")}', fontweight='bold')
    ax.set_ylabel('Students')
    ax.legend(fontsize=8)

# Plot 2: Absences by risk level — both years
ax = axes[1, 0]
for year, marker in [('24_25', 'o'), ('25_26', 's')]:
    df_y = q4_pd[q4_pd['year'] == year].sort_values('avg_grade')
    ax.plot(df_y['avg_absences'], df_y['avg_grade'],
            marker=marker, linewidth=2, markersize=8,
            label=year.replace('_', '-'))
    for _, row in df_y.iterrows():
        ax.annotate(f"{row['risk_level']}\n({year[-2:]})",
                   (row['avg_absences'], row['avg_grade']),
                   textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.set_xlabel('Average absences')
ax.set_ylabel('Average grade')
ax.set_title('Absences vs Grade by Risk Level', fontweight='bold')
ax.legend(fontsize=8)
ax.axhline(y=4.0, color='red', linestyle='--', alpha=0.5, label='Pass threshold')

# Plot 3: Subject averages comparison
ax = axes[1, 1]
q3_pd = spark.read.parquet(f"{SILVER}/eda_subject_performance").toPandas()
subjects = ['Science', 'Mathematics', 'English', 'Lengua_Castellana',
            'Mandarin', 'Financial_Maths', 'ICT_STEM', 'Physical_Education']
x = range(len(subjects))
width = 0.35
for i, year in enumerate(['24_25', '25_26']):
    row = q3_pd[q3_pd['year'] == year].iloc[0]
    vals = [float(row[s]) if s in row and row[s] is not None else 0 
            for s in subjects]
    ax.bar([xi + i*width for xi in x], vals, width=width,
           label=year.replace('_', '-'),
           color=['#3498db', '#e67e22'][i], alpha=0.8)
ax.set_xticks([xi + width/2 for xi in x])
ax.set_xticklabels([s.replace('_',' ')[:8] for s in subjects],
                   rotation=45, ha='right', fontsize=8)
ax.axhline(y=4.0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Average Grade by Subject — Year Comparison', fontweight='bold')
ax.set_ylabel('Average grade')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/eda_sql_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ EDA visualizations saved")
print(f"\n✓ Notebook 02b complete — EDA results saved to Silver")